In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

### Prompt Template

In [2]:
my_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a movie summarizer."),
        ("user", "Please explain the movie in breif: {topic}")
    ]
)
my_template

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a movie summarizer.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Please explain the movie in breif: {topic}'), additional_kwargs={})])

##### LLM

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

#### Parser

In [4]:
str_parser = StrOutputParser()

In [7]:
from langchain_core.runnables import RunnableLambda

def dictionary_maker(text: str) -> dict:
    return {"text" : text}

dictionary_maker_runnable = RunnableLambda(dictionary_maker)

#### Chain 1 

In [9]:
#### Task 1: Prompt
prompt_post = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a LinkedIn post generator."),
        ("user", "Create a post for the following text for LinkedIn: {text}")
    ]
)

# Task 2: LLM
llm_generate = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Task 3: Parser
str_parser = StrOutputParser()

chain_linkedin = prompt_post | llm_generate | str_parser

#### Chain 2 (using a function)

In [10]:
def insta_chain(text : dict): 
    
    text = text["text"]
    
    #### Task 1: Prompt
    prompt_post = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a Instagram post generator."),
            ("user", "Create a post for the following text for Instagram: {text}")
        ]
    )

    # Task 2: LLM
    llm_generate = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

    # Task 3: Parser
    str_parser = StrOutputParser()
    
    chain_insta = prompt_post | llm_generate | str_parser
    response = chain_insta.invoke(text)
    return response


insta_chain_runnable = RunnableLambda(insta_chain)

#### Final Orchestration

In [11]:
from langchain_core.runnables import RunnableParallel

final_chain = (
    my_template | 
    llm | 
    str_parser | 
    dictionary_maker_runnable |
    RunnableParallel(branches = {
        "linkedin": chain_linkedin, 
        "instagram": insta_chain_runnable
        })
)

In [12]:
final_chain.invoke("KGF")

{'branches': {'linkedin': 'Here are a few options for a LinkedIn post about KGF, playing on different angles:\n\n---\n\n**Option 1 (Focus on Leadership & Ambition):**\n\n**Subject: Beyond the Action: KGF\'s Masterclass in Ambition and Unconventional Leadership**\n\nKGF (Kolar Gold Fields) isn\'t just an epic Indian action film; it\'s a compelling study in relentless ambition and the rise of an unconventional leader.\n\nRocky\'s journey, fueled by a fierce promise to his dying mother to die rich and powerful, showcases an extreme drive from the slums of Mumbai to infiltrating the brutal Kolar Gold Fields. His strategic move to gain the trust and loyalty of enslaved miners, orchestrating a massive rebellion to overthrow Garuda, offers a fascinating (and violent) look at:\n\n*   **Strategic Infiltration:** How to dismantle power structures from within.\n*   **Building Loyalty:** Earning trust among the oppressed.\n*   **Visionary Drive:** The single-minded pursuit of a powerful objective.

#### Chain as a Runnable

In [16]:
### Task 1: Beautify  function

def beautify(final_res: dict) -> dict:
    linkedin_res = final_res["branches"]["linkedin"]
    insta_res = final_res["branches"]["instagram"]
    
    return {"linkedin": linkedin_res, "instagram": insta_res}


beautify_runnable = RunnableLambda(beautify)

#### Task 2
beautified_chain = final_chain | beautify_runnable
beautified_chain.invoke("Pushpa")

{'linkedin': 'Here are a few options for a LinkedIn post, choose the one that best fits your desired tone:\n\n---\n\n**Option 1: Focus on Ambition & Strategy (Business Analogy)**\n\n🚀 Ever wondered what it takes to disrupt an entire industry and climb the ranks from the ground up?\n\n"Pushpa: The Rise" introduces us to Pushpa Raj, a character driven by an unyielding ambition and a fierce desire to prove his worth. Despite his humble beginnings and the stigma of his illegitimate birth, Pushpa transforms from a mere coolie into a key player in the illegal red sanders trade.\n\nHis journey is a fascinating (albeit illicit) case study in:\n*   **Strategic Thinking:** Outsmarting rivals and navigating complex power structures.\n*   **Resourcefulness:** Using wit and courage to overcome obstacles.\n*   **Leadership (Unconventional):** Building an empire and challenging established hierarchies.\n*   **Motivation:** The relentless pursuit of respect and power.\n\nWhile his methods are certainl